# מפגש 10 — פתרון מורה מלא עם בדיקות
כל המספרים נבדקים מול מקור Penguins ונתוני שש העסקאות מהמעבדה, לא מועתקים מדוגמת המורה הקטנה.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
url='https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv'
penguins_df=pd.read_csv(url)
assert penguins_df.shape==(344,7)

## חלק 1 — טבלת איים: count של מדידות ידועות לעומת size שורות

In [ ]:
mean_flipper_by_island=penguins_df.groupby('island')['flipper_length_mm'].mean()
known_flippers_by_island=penguins_df.groupby('island')['flipper_length_mm'].count()
all_rows_by_island=penguins_df.groupby('island').size()
summary_islands=penguins_df.groupby('island')['flipper_length_mm'].agg(['size','count','mean','median'])
display(summary_islands)
assert all_rows_by_island.to_dict()=={'Biscoe':168,'Dream':124,'Torgersen':52}
assert known_flippers_by_island.to_dict()=={'Biscoe':167,'Dream':124,'Torgersen':51}
assert np.isclose(mean_flipper_by_island['Biscoe'],35021/167)
assert np.isclose(mean_flipper_by_island['Dream'],23941/124)
assert np.isclose(mean_flipper_by_island['Torgersen'],9751/51)

## שני גרפים לפי שתי שאלות מחקריות
Countplot סופרת שורות, Barplot estimator='mean' מציגה ממוצע של מדידות ידועות. errorbar=None לא מוכיח הבדל מובהק.

In [ ]:
island_order=['Biscoe','Dream','Torgersen']
plt.figure(figsize=(7,4))
sns.countplot(data=penguins_df,x='island',order=island_order,color='teal')
plt.title('Observations by island')
plt.xlabel('Island')
plt.ylabel('Number of observations')
plt.tight_layout()
plt.show()
plt.figure(figsize=(7,4))
sns.barplot(data=penguins_df,x='island',y='flipper_length_mm',order=island_order,estimator='mean',errorbar=None,color='steelblue')
plt.title('Mean flipper length by island')
plt.xlabel('Island')
plt.ylabel('Mean flipper length (mm)')
plt.tight_layout()
plt.show()
assert summary_islands['size'].sum()==344
assert summary_islands['count'].sum()==342
assert penguins_df.shape==(344,7)

## חלק 2 — שש עסקאות; ארבע אגרגציות
כמות חסרה אינה אפס יחידות. כל שורה בתוצאת GroupBy מסכמת חנות.

In [ ]:
sales=pd.DataFrame({'branch':['A','A','B','B','B','C'],'units':[1.,np.nan,2.,2.,np.nan,5.]})
grouped=sales.groupby('branch')['units']
summary=grouped.agg(['size','count','sum','mean'])
display(summary)
assert summary['size'].to_dict()=={'A':2,'B':3,'C':1}
assert summary['count'].to_dict()=={'A':1,'B':2,'C':1}
assert summary['sum'].to_dict()=={'A':1.,'B':4.,'C':5.}
assert summary['mean'].to_dict()=={'A':1.,'B':2.,'C':5.}
assert len(sales)==6 and sales['units'].count()==4

## ממוצע כולל לעומת ממוצע ממוצעי קבוצות
בממוצע הכולל ארבע המדידות הידועות מקבלות משקל שווה; בממוצע הקבוצות שלוש החנויות מקבלות משקל שווה.

In [ ]:
overall_mean=sales['units'].mean()
mean_of_group_means=summary['mean'].mean()
print('Overall mean',overall_mean,'Mean of group means',mean_of_group_means)
assert np.isclose(overall_mean,2.5)
assert np.isclose(mean_of_group_means,8/3)
assert not np.isclose(overall_mean,mean_of_group_means)

## מקרה גבול — כל הערכים חסרים
ברירת המחדל של sum עשויה להציג 0; שיטת min_count=1 שומרת את המשמעות של 'לא ידוע'.

In [ ]:
missing=pd.DataFrame({'branch':['D','D'],'units':[np.nan,np.nan]})
g=missing.groupby('branch')['units']
assert g.size()['D']==2
assert g.count()['D']==0
assert g.sum()['D']==0
assert np.isnan(g.sum(min_count=1)['D'])
assert np.isnan(g.mean()['D'])

## סיכום מורה
אין לפרש מיון או GroupBy כאימון מודל; אין להסיק שהאי גרם לאורך סנפיר, אין לפרש ממוצע כערך כל פרט, ואין להכניס סיכומי העתיד לקלט חיזוי מן העבר. לבגרות/הגנה דרשו פירוש של פלט, יחידות ומגבלה.